# Mechanistic Interpretability: One Neuron, Two Features

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/mechanistic_interpretability_one_neuron.ipynb)

Companion notebook to [the blog post](https://sesen.ai/blog/mechanistic-interpretability-one-neuron-two-features).

Three probes, pointed at one unit of a small MNIST CNN:

1. **Dataset examples**: which real inputs excite it most
2. **Activation maximisation**: which synthetic input excites it most
3. **Ablation**: what the network loses when it is switched off

They disagree, and the disagreement is the point. Along the way you will find
the unit that fires for both zeros and twos, the unit that fires for nothing at
all, and the reason single-unit ablation reports almost no damage.

Everything is CPU-only. Training takes about a minute; every probe after that
takes seconds.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

torch.manual_seed(0)

MNIST_MEAN, MNIST_STD = 0.1307, 0.3081
PIXEL_MIN = (0.0 - MNIST_MEAN) / MNIST_STD
PIXEL_MAX = (1.0 - MNIST_MEAN) / MNIST_STD

transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))])
train_data = datasets.MNIST('data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('data', train=False, download=True, transform=transform)

def denorm(x):
    return np.clip(np.asarray(x) * MNIST_STD + MNIST_MEAN, 0, 1)

## 1. The network

Straight from [CNNs from Scratch](https://sesen.ai/blog/convolutional-neural-networks-from-scratch):
three stride-2 convolutions of 8, 16 and 32 channels, one linear head, Adam at
1e-3, batch size 64, five epochs.

`features()` takes a `pre_relu` flag. Activation maximisation needs it: a unit
sitting at zero for the initial noise image has an exactly zero gradient through
the ReLU and can never climb out.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=5, padding=2, stride=2)   # 28->14
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1, stride=2)  # 14->7
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1, stride=2) # 7->4
        self.fc = nn.Linear(32 * 4 * 4, 10)

    def features(self, x, upto=3, pre_relu=False):
        x = self.conv1(x)
        if upto == 1:
            return x if pre_relu else F.relu(x)
        x = self.conv2(F.relu(x))
        if upto == 2:
            return x if pre_relu else F.relu(x)
        x = self.conv3(F.relu(x))
        return x if pre_relu else F.relu(x)

    def forward(self, x):
        h = self.features(x)
        return self.fc(h.view(x.size(0), -1))


model = SimpleCNN()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1000)

for epoch in range(5):
    model.train()
    for images, labels in train_loader:
        loss = F.cross_entropy(model(images), labels)
        optimiser.zero_grad(); loss.backward(); optimiser.step()
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            correct += (model(images).argmax(1) == labels).sum().item()
    print(f"Epoch {epoch + 1}: {correct / len(test_data) * 100:.1f}% accuracy")

## 2. What counts as one unit

A conv layer has no neurons in the dense-layer sense. Each of the 32 outputs of
`conv3` is a channel: one 3x3 filter applied at all 16 positions of a 4x4 grid.
A unit's activation on an image is that channel's mean over the 16 positions.

Stacking three stride-2 layers gives each `conv3` position a 17x17 window on the
input, stepping 8 pixels between neighbours. When we look at what excites a unit
we crop to that window, because showing the whole digit lets the eye fill in
structure the unit never saw.

In [ ]:
RF_SIZE, RF_STRIDE, RF_OFFSET = 17, 8, -6

def receptive_field_patch(image, row, col):
    top, left = RF_OFFSET + RF_STRIDE * row, RF_OFFSET + RF_STRIDE * col
    patch = np.full((RF_SIZE, RF_SIZE), PIXEL_MIN, dtype=np.float32)
    st, sl = max(top, 0), max(left, 0)
    sb, sr = min(top + RF_SIZE, 28), min(left + RF_SIZE, 28)
    patch[st - top:sb - top, sl - left:sr - left] = image[st:sb, sl:sr]
    return patch


mean_acts, positions, labels, images = [], [], [], []
model.eval()
with torch.no_grad():
    for batch, target in torch.utils.data.DataLoader(test_data, batch_size=500):
        acts = model.features(batch)                  # [B, 32, 4, 4]
        mean_acts.append(acts.mean(dim=(2, 3)))
        flat = acts.view(len(batch), 32, 16).argmax(dim=2)
        positions.append(torch.stack([flat // 4, flat % 4], dim=-1))
        labels.append(target); images.append(batch)

mean_acts = torch.cat(mean_acts).numpy()
positions = torch.cat(positions).numpy()
labels = torch.cat(labels).numpy()
images = torch.cat(images).numpy()
print(mean_acts.shape)

## 3. Probe 1: dataset examples

Sort the test set by a unit's activation and look at the top. No optimisation,
nothing that can hallucinate: these are inputs the network met in the wild.

In [ ]:
def top_k(channel, k=100):
    return np.argsort(-mean_acts[:, channel])[:k]

for unit in (25, 21):
    hist = np.bincount(labels[top_k(unit)], minlength=10)
    print(f"unit {unit}: {hist.tolist()}")

fig, axes = plt.subplots(2, 9, figsize=(11, 2.8))
for row, unit in enumerate((25, 21)):
    for j, idx in enumerate(top_k(unit)[:9]):
        axes[row, j].imshow(denorm(images[idx, 0]), cmap='gray', vmin=0, vmax=1)
        axes[row, j].set_title(str(labels[idx]), fontsize=8)
        axes[row, j].axis('off')
    axes[row, 0].set_ylabel(f"unit {unit}")
fig.suptitle("Top 9 activating test digits: unit 25 (top), unit 21 (bottom)")
plt.show()

Unit 25's top 100 is 44 zeros and 43 twos. Unit 21's is 89 zeros. One of those
is a digit detector and the other is something else.

Note the `k`. At `k = 9`, the size of the grid above, unit 25 shows six zeros and
three twos and reads as a noisy zero detector. The split only becomes obvious in
the hundred-image histogram.

## 4. Probe 2: activation maximisation

Freeze the weights and run gradient ascent on the input. Three regularisers keep
the result legible, all from Olah et al. (2017): random jitter before each
forward pass, a weak L2 pull towards the mean pixel, and a periodic blur that
suppresses the high-frequency checkerboard unconstrained ascent always finds.

In [ ]:
def blur(x, weight=0.3):
    k = torch.tensor([[1., 2., 1.], [2., 4., 2.], [1., 2., 1.]]).view(1, 1, 3, 3)
    k = k / k.sum()
    return (1 - weight) * x + weight * F.conv2d(F.pad(x, (1, 1, 1, 1), mode='replicate'), k)


def activation_maximise(channel, layer=3, steps=320, lr=0.1, jitter=2, l2=1e-3,
                        seed=0, position=None, snapshot_every=None):
    g = torch.Generator().manual_seed(seed)
    img = (torch.randn(1, 1, 28, 28, generator=g) * 0.1).requires_grad_(True)
    opt = torch.optim.Adam([img], lr=lr)

    def score(x, pre_relu):
        maps = model.features(x, upto=layer, pre_relu=pre_relu)[0, channel]
        return maps[position] if position is not None else maps.mean()

    trace, snaps = [], []
    for step in range(steps):
        if position is None:
            shift = torch.randint(-jitter, jitter + 1, (2,), generator=g)
            x = torch.roll(img, tuple(shift.tolist()), dims=(2, 3))
        else:
            x = img
        loss = -score(x, pre_relu=True) + l2 * img.pow(2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            if (step + 1) % 4 == 0:
                img.data = blur(img.data)
            img.data = img.data.clamp(PIXEL_MIN, PIXEL_MAX)
            trace.append(float(score(img, pre_relu=False)))
        if snapshot_every and step % snapshot_every == 0:
            snaps.append(img.detach().numpy()[0, 0].copy())
    return img.detach().numpy()[0, 0], np.array(trace), snaps


am25, trace25, snaps25 = activation_maximise(25, snapshot_every=16)

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.2), gridspec_kw={'width_ratios': [1, 1.6]})
a.imshow(denorm(am25), cmap='gray', vmin=0, vmax=1); a.axis('off')
a.set_title("unit 25, synthesised")
b.plot(trace25); b.set_xlabel("gradient ascent step"); b.set_ylabel("activation")
plt.show()

Diagonal stripes. Unit 25's real favourites, from probe 1, are zeros and twos.
The picture is confident, legible, and it describes neither of them.

**Try this:** set `l2=0` and remove the blur, then look again. The activation
goes higher and the image stops meaning anything. That failure mode is why the
regularisers exist.

### The first layer, where the probes agree

A convolution is linear before its ReLU, so maximising the *mean* over every
spatial position drives all 784 pixels to the clamp and returns a saturated
tile. Score a single position instead and the 5x5 receptive field there is the
whole answer, which we can compare against the learned kernel directly.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(11, 3.2))
W = model.conv1.weight.detach().numpy()[:, 0]
for c in range(8):
    img, _, _ = activation_maximise(c, layer=1, steps=200, position=(7, 7), l2=1e-2)
    patch = img[12:17, 12:17]
    r = np.corrcoef(patch.ravel(), W[c].ravel())[0, 1]
    axes[0, c].imshow(W[c], cmap='RdBu_r', vmin=-np.abs(W[c]).max(), vmax=np.abs(W[c]).max())
    axes[0, c].set_title(f"unit {c}", fontsize=9); axes[0, c].axis('off')
    axes[1, c].imshow(denorm(patch), cmap='gray', vmin=0, vmax=1)
    axes[1, c].set_title(f"r = {r:.2f}", fontsize=9); axes[1, c].axis('off')
fig.suptitle("conv1: learned kernel (top) vs what gradient ascent settles on (bottom)")
plt.show()

At depth one the synthesised picture and the ground truth are the same object.
That is why first-layer feature visualisations are so persuasive, and why people
extrapolate from them to depths where the agreement has gone.

## 5. Probe 3: ablation

The only causal probe. Zero the channel and measure what the network loses.

In [ ]:
def per_class_accuracy(ablate=None):
    correct, total = np.zeros(10), np.zeros(10)
    with torch.no_grad():
        for imgs, labs in torch.utils.data.DataLoader(test_data, batch_size=1000):
            acts = model.features(imgs)
            if ablate is not None:
                acts = acts.clone(); acts[:, ablate] = 0.0
            preds = model.fc(acts.view(len(imgs), -1)).argmax(1)
            for cls in range(10):
                mask = labs == cls
                total[cls] += mask.sum().item()
                correct[cls] += (preds[mask] == cls).sum().item()
    return correct / total


baseline = per_class_accuracy()
drops = np.array([baseline - per_class_accuracy(ablate=c) for c in range(32)])
print(f"baseline {baseline.mean() * 100:.2f}%")
print(f"largest single-unit per-class cost: {drops.max() * 100:.2f} points")
print(f"unit 25 per-class cost (points): {np.round(drops[25] * 100, 2).tolist()}")

plt.figure(figsize=(10, 3.6))
plt.imshow(drops.T * 100, cmap='RdBu_r', vmin=-np.abs(drops).max() * 100,
           vmax=np.abs(drops).max() * 100, aspect='auto')
plt.colorbar(label='accuracy lost (points)')
plt.xlabel('unit switched off'); plt.ylabel('digit class'); plt.yticks(range(10))
plt.show()

Almost nothing anywhere. That cannot be true of all 32 units at once, so the
explanation must be redundancy: switch off one unit and the rest of the layer
covers for it. Group ablation shows it directly.

In [ ]:
preferred = np.array([np.bincount(labels[top_k(c)], minlength=10).argmax()
                      for c in range(32)])

def group_ablate(channels):
    correct, total = np.zeros(10), np.zeros(10)
    with torch.no_grad():
        for imgs, labs in torch.utils.data.DataLoader(test_data, batch_size=1000):
            acts = model.features(imgs).clone(); acts[:, channels] = 0.0
            preds = model.fc(acts.view(len(imgs), -1)).argmax(1)
            for cls in range(10):
                mask = labs == cls
                total[cls] += mask.sum().item()
                correct[cls] += (preds[mask] == cls).sum().item()
    return correct / total


rng = np.random.default_rng(0)
print(f"{'digit':>5} {'k':>3} {'group':>8} {'random k':>9}")
for cls in range(10):
    group = np.where(preferred == cls)[0]
    if len(group) < 2:
        continue
    grouped = (baseline[cls] - group_ablate(group)[cls]) * 100
    ctrl = np.mean([(baseline[cls] - group_ablate(rng.choice(32, len(group), replace=False))[cls]) * 100
                    for _ in range(3)])
    print(f"{cls:>5} {len(group):>3} {grouped:>8.2f} {ctrl:>9.2f}")

Groups of five or more cost several points where a size-matched random group
costs nothing. A single-unit ablation measures how well the rest of the network
copes without that unit, which is a real quantity and not the one most readers
think they are measuring.

## 6. One neuron, two features

Take the 100 receptive-field patches where a unit fired hardest and split them in
two. k-means always returns two clusters, so the split alone proves nothing: the
question is how far apart they land, against a unit we know is class-selective.

In [ ]:
def two_means(x, seed=0, iters=50):
    rng = np.random.default_rng(seed)
    flat = x.reshape(len(x), -1)
    centres = flat[rng.choice(len(flat), 2, replace=False)].copy()
    lab = np.zeros(len(flat), dtype=int)
    for _ in range(iters):
        new = ((flat[:, None] - centres[None]) ** 2).sum(-1).argmin(1)
        if (new == lab).all():
            break
        lab = new
        for k in range(2):
            if (lab == k).any():
                centres[k] = flat[lab == k].mean(0)
    return lab, centres.reshape(2, *x.shape[1:])


def split_report(unit):
    order = top_k(unit)
    patches = np.stack([receptive_field_patch(images[i, 0], *positions[i, unit]) for i in order])
    lab, centres = two_means(patches)
    a, b = centres[0].ravel() - centres[0].mean(), centres[1].ravel() - centres[1].mean()
    corr = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
    for k in range(2):
        h = np.bincount(labels[order][lab == k], minlength=10)
        print(f"  cluster {k + 1}: {h.sum():>3} patches, {h.max() / h.sum():.0%} are {h.argmax()}s")
    print(f"  centroid correlation: {corr:.2f}")
    return order, patches, lab, centres


for unit in (25, 21):
    print(f"unit {unit}:")
    order, patches, lab, centres = split_report(unit)
    fig, axes = plt.subplots(2, 8, figsize=(10, 2.7))
    for k in range(2):
        axes[k, 0].imshow(denorm(centres[k]), cmap='gray', vmin=0, vmax=1)
        axes[k, 0].set_title('mean', fontsize=8); axes[k, 0].axis('off')
        for j, i in enumerate(np.where(lab == k)[0][:7]):
            axes[k, j + 1].imshow(denorm(patches[i]), cmap='gray', vmin=0, vmax=1)
            axes[k, j + 1].set_title(str(labels[order[i]]), fontsize=8)
            axes[k, j + 1].axis('off')
    fig.suptitle(f"unit {unit}: top-100 receptive fields, split in two")
    plt.show()

Unit 25 splits into a rounded arc that is 81% zeros and a flat base stroke that
is 82% twos, with the cluster means correlating at 0.41. Unit 21 gives back the
same arc twice, correlating at 0.79.

That is polysemanticity. Elhage et al. (2022) explain it as superposition: a
layer with 32 channels can hold 32 features cleanly, and MNIST digits are built
from more than 32 distinguishable strokes, so the layer stores them as
directions that are almost but not quite orthogonal.

It also explains probe 2. Gradient ascent maximises a scalar, so it returns one
input, and a unit with two preferred inputs has at least two optima. Unit 25's
stripes are a compromise between an arc and a corner.

## 7. The dead unit

One unit needs no interpretation at all.

In [ ]:
firing = (mean_acts > 0).mean(axis=0)
dead = np.where(firing < 0.01)[0]
print("units firing on under 1% of test images:", dead.tolist())

for c in dead:
    img, trace, _ = activation_maximise(int(c))
    print(f"unit {c}: activation after 320 steps = {trace[-1]:.4f}, "
          f"worst per-class ablation cost = {drops[c].max() * 100:.2f} points")
    plt.figure(figsize=(2.2, 2.2))
    plt.imshow(denorm(img), cmap='gray', vmin=0, vmax=1); plt.axis('off')
    plt.title(f"unit {c}, synthesised", fontsize=9)
    plt.show()

The optimiser always returns an image, including for a unit that fires on none
of the 10,000 test digits. Skim a grid of 32 feature visualisations at thumbnail
size and this one does not stand out. The activation value printed beside it
does.

## 8. All three probes, all 32 units

Reduce each probe to the digit class it names, then count the agreements.

In [ ]:
def looks_like(am_image, k=25):
    a = am_image.ravel() - am_image.mean()
    b = images.reshape(len(images), -1)
    b = b - b.mean(1, keepdims=True)
    corr = (b @ a) / (np.linalg.norm(b, axis=1) * np.linalg.norm(a) + 1e-9)
    return int(np.bincount(labels[np.argsort(-corr)[:k]], minlength=10).argmax())


data_v = [int(np.bincount(labels[top_k(c)], minlength=10).argmax()) for c in range(32)]
am_v = [looks_like(activation_maximise(c)[0]) for c in range(32)]
abl_v = [int(drops[c].argmax()) for c in range(32)]

live = [c for c in range(32) if c not in dead]
print(f"over {len(live)} live units:")
print("  dataset == ablation :", sum(data_v[c] == abl_v[c] for c in live))
print("  dataset == act-max  :", sum(data_v[c] == am_v[c] for c in live))
print("  all three agree     :", sum(data_v[c] == am_v[c] == abl_v[c] for c in live))

## Exercises

1. **Widen the layer.** Change `conv3` to 64 channels, retrain, and rerun the
   two-means split. Does the fraction of units clearing the polysemantic bar
   fall? Superposition predicts it should, without disappearing.
2. **Narrow it.** Drop `conv3` to 16 channels and check the same number moves
   the other way. This is the cheaper half of the experiment.
3. **Break activation maximisation on purpose.** Set `pre_relu=False` in the
   score function and confirm every trace stays flat at zero. That bug is easy
   to write and produces plausible-looking output.
4. **Change the split.** Replace two-means with three-means and look for a unit
   holding three features. The two-cluster choice here is a floor, not a finding.
5. **Fit a sparse autoencoder** to the 32-dimensional activation vectors with an
   overcomplete dictionary and an L1 penalty. Check whether the zero-arc and the
   two-base separate into different dictionary elements. This is the toy version
   of what Templeton et al. (2024) ran on Claude 3 Sonnet.

## Further reading

- Olah, Mordvintsev and Schubert (2017), [Feature Visualization](https://distill.pub/2017/feature-visualization/)
- Bau et al. (2017), [Network Dissection](https://netdissect.csail.mit.edu/)
- Olah et al. (2020), [Zoom In: An Introduction to Circuits](https://distill.pub/2020/circuits/zoom-in/)
- Elhage et al. (2022), [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html)
- Templeton et al. (2024), [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/)